### filter the dataset by selecting only the data samples where “price_range” = 1.


In [1]:
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Load dataset
csv_path = "../mobile_price.csv"
df = pd.read_csv(csv_path)

# Filter samples where price_range = 1
df_q3 = df[df["price_range"] == 1].copy()

# Select required columns
selected_features = ["ram", "int_memory", "px_width", "battery_power"]
df_q3_features = df_q3[selected_features].copy()

print("Number of samples with price_range = 1:", len(df_q3_features))
df_q3_features.head()

Number of samples with price_range = 1: 500


,ram,int_memory,px_width,battery_power
0,2549,7,756,842
4,1411,44,1212,1821
5,1067,22,1654,1859
12,1482,33,748,1815
18,1835,49,878,1131


### divide the range into three intervals using a 3:4:3 ratio: low, medium, and high.

In [2]:
def categorize_343(series):
    min_val = series.min()
    max_val = series.max()
    value_range = max_val - min_val
    
    low_threshold = min_val + 0.3 * value_range
    high_threshold = min_val + 0.7 * value_range
    
    def categorize_value(x):
        if x <= low_threshold:
            return "low"
        elif x <= high_threshold:
            return "medium"
        else:
            return "high"
    
    return series.apply(categorize_value), min_val, max_val, low_threshold, high_threshold


categorized_df = pd.DataFrame()
feature_ranges = []

for feature in selected_features:
    categorized_df[feature], min_val, max_val, low_th, high_th = categorize_343(df_q3_features[feature])
    feature_ranges.append({
        "feature": feature,
        "min": min_val,
        "max": max_val,
        "low_threshold": low_th,
        "high_threshold": high_th
    })

feature_ranges_df = pd.DataFrame(feature_ranges)

print("Feature ranges and thresholds:")
display(feature_ranges_df)

print("Categorized data:")
display(categorized_df.head())

Feature ranges and thresholds:


,feature,min,max,low_threshold,high_threshold
0,ram,387,2811,1114.2,2083.8
1,int_memory,2,64,20.6,45.4
2,px_width,500,1998,949.4,1548.6
3,battery_power,501,1996,949.5,1547.5


Categorized data:


,ram,int_memory,px_width,battery_power
0,high,low,low,low
4,medium,medium,medium,high
5,low,medium,high,high
12,medium,medium,low,high
18,medium,high,low,medium


### transaction format

In [3]:
transactions = []

for _, row in categorized_df.iterrows():
    transaction = []
    for feature in selected_features:
        transaction.append(f"{feature}_{row[feature]}")
    transactions.append(transaction)

transactions[:5]

[['ram_high', 'int_memory_low', 'px_width_low', 'battery_power_low'],
 ['ram_medium', 'int_memory_medium', 'px_width_medium', 'battery_power_high'],
 ['ram_low', 'int_memory_medium', 'px_width_high', 'battery_power_high'],
 ['ram_medium', 'int_memory_medium', 'px_width_low', 'battery_power_high'],
 ['ram_medium', 'int_memory_high', 'px_width_low', 'battery_power_medium']]

### One-hot encoding transactions

In [4]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

transaction_df = pd.DataFrame(te_array, columns=te.columns_)

transaction_df.head()

,battery_power_high,battery_power_low,battery_power_medium,int_memory_high,int_memory_low,int_memory_medium,px_width_high,px_width_low,px_width_medium,ram_high,ram_low,ram_medium
0,False,True,False,False,True,False,False,True,False,True,False,False
1,True,False,False,False,False,True,False,False,True,False,False,True
2,True,False,False,False,False,True,True,False,False,False,True,False
3,True,False,False,False,False,True,False,True,False,False,False,True
4,False,False,True,True,False,False,False,True,False,False,False,True


### list all frequent patterns showing support ≥ 0.3 using FP-growth

In [10]:
frequent_patterns = fpgrowth(
    transaction_df,
    min_support=0.3,
    use_colnames=True
)

# 依 support 由高到低排序
frequent_patterns = frequent_patterns.sort_values(
    by="support",
    ascending=False
).reset_index(drop=True)

frequent_patterns_report = frequent_patterns.copy()
frequent_patterns_report["itemsets"] = frequent_patterns_report["itemsets"].apply(
    lambda x: ", ".join(sorted(list(x)))
)

frequent_patterns_report

,support,itemsets
0,0.682,ram_medium
1,0.416,px_width_medium
2,0.414,battery_power_medium
3,0.412,int_memory_medium
4,0.318,"battery_power_medium, ram_medium"
5,0.316,int_memory_low
6,0.308,battery_power_low
7,0.306,"px_width_medium, ram_medium"


### list all the association rules where support ≥ 0.3, confidence ≥ 0.4 and lift ≥ 0.8

In [12]:
rules = association_rules(
    frequent_patterns,
    metric="confidence",
    min_threshold=0.4
)

# Filter rules by support, confidence, and lift
rules_filtered = rules[
    (rules["support"] >= 0.3) &
    (rules["confidence"] >= 0.4) &
    (rules["lift"] >= 0.8)
].copy()

# Sort rules
rules_filtered = rules_filtered.sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
).reset_index(drop=True)

# Convert frozenset to readable text
rules_report = rules_filtered.copy()

rules_report["antecedents"] = rules_report["antecedents"].apply(
    lambda x: ", ".join(sorted(list(x)))
)

rules_report["consequents"] = rules_report["consequents"].apply(
    lambda x: ", ".join(sorted(list(x)))
)

rules_report = rules_report[
    ["antecedents", "consequents", "support", "confidence", "lift"]
]

rules_report

,antecedents,consequents,support,confidence,lift
0,battery_power_medium,ram_medium,0.318,0.768116,1.126270
1,ram_medium,battery_power_medium,0.318,0.466276,1.126270
2,px_width_medium,ram_medium,0.306,0.735577,1.078559
3,ram_medium,px_width_medium,0.306,0.448680,1.078559
